In [4]:
import pandas as pd
import os
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain
from langchain_google_genai import ChatGoogleGenerativeAI
from itertools import cycle
import time
import re

In [ ]:
import os
from dotenv import load_dotenv   # pip install python-dotenv
load_dotenv()
DEEPSEEK_API_KEY= os.getenv("DEEPSEEK_API_KEY")

In [ ]:
from langchain_deepseek import ChatDeepSeek

MODEL_NAME = "deepseek-flash"  # or "deepseek-reasoner"

TEMPERATURE = 0

llm = ChatDeepSeek(
    model=MODEL_NAME,
    temperature=TEMPERATURE,
    api_key=DEEPSEEK_API_KEY
)

In [ ]:
template_positive_personal =  """
Rewrite the job description below as one flowing paragraph in plain prose — no headers, no bullet points, and no inline list-labels like "Responsibilities:" or "Requirements include:" followed by a semicolon list. Write connected sentences, not a bulleted list with the line breaks removed.

Keep everything specific to this role — responsibilities, required and preferred qualifications, tools/technologies, domain, seniority, experience, education, and any condition that affects whether someone qualifies — no matter which section it sits under. 

Job description:

{job_description}

**Your Reply:**
"""

In [ ]:

template_positive_personal = """
You are an expert technical recruiter and Applicant Tracking System (ATS) parser. I will provide you with a job description. Your task is to analyze the text and extract the job title, all hard skills/tools, and the top soft skills.

### Instructions:
1. **title**: Extract the exact job title.

2. **Hard Skills (Exhaustive, Normalized & Inclusive)**:
   - Extract EVERY technical requirement, programming language, framework, library, cloud platform, AI model, software, developer tool, methodology, and domain expertise mentioned in the text.
   - IMPORTANT CAPTURE RULE: Extract all named software, platforms, and tools regardless of the context they are used in. Do not filter out tools just because the JD mentions them as "examples," "nice-to-haves," or in a negative/restrictive context (e.g., "experience beyond just using [Tool]").
   - CRITICAL NORMALIZATION: Standardize skill names into resume-ready keywords by stripping away unnecessary descriptive filler words (e.g., remove words like "practices", "techniques", "frameworks", "environments", "applications", "methodologies", or "development"). 
   - Normalization Examples: Output "Agile" instead of "Agile methodologies", "Frontend" instead of "frontend frameworks", and keep exact tool names as proper nouns.

3. **Soft Skills**:

   -  Interpersonal traits (Agile methodology, Cross-functional collaboration, Mentorship, Adaptability).

### Output Format:
Return the output EXACTLY in the following JSON format. Do not include any extra text, explanations, or markdown formatting outside of the JSON block.

{{
  "title": "Extracted Job Title",
  "Hard Skills": ["Skill 1", "Skill 2", "...", "Skill N"],
  "Soft Skills": ["Skill 1", "Skill 2", "Skill 3", "Skill 4"]
}}

Job description: 
{job_description}

"""


prompt_template = PromptTemplate(
            input_variables=job_description, 
            template=template_positive_personal 
)
# 1. Create the chain object correctly (without the extra comma making it a tuple)
chain = LLMChain(llm=llm, prompt=prompt_template)

# 2. Run the chain with the input variable defined in the template
# The template uses {review}, so we pass the variable 'review' with its content.
res = chain.run(job_description=job_description) 

# 3. Print the result
print(res)

In [ ]:
import json
from langchain_deepseek import ChatDeepSeek
from langchain_core.prompts import PromptTemplate

# Initialize model
MODEL_NAME = "deepseek-flash"  # or "deepseek-reasoner"
TEMPERATURE = 0

llm = ChatDeepSeek(
    model=MODEL_NAME,
    temperature=TEMPERATURE,
    api_key=DEEPSEEK_API_KEY
)

# DeepSeek-Optimized Prompt Template
prompt_template = PromptTemplate.from_template("""
<system_instructions>
You are an expert Executive Technical Resume Strategist. 
Your task is to rewrite a candidate's CV components (experience, title, bullets) to match a target Job Description with human easy to understad uk english.

CRITICAL RULES:
1. TRUTHFUL RE-FRAMING: Do NOT invent experience or tools not present in the source. Re-frame real experience or add experience which is present or relevant in the profile.
 
2. LENGTH CONSTRAINTS (CRITICAL): 
   - you can resuffle the project but need to Keep the total number of projects and the number of bullets per project roughly the same as the current input.

3. STRATEGIC BOLDING: Use Markdown bolding (`**text**`) VERY sparingly to draw the recruiter's eye. 

4. OUTPUT FORMAT: Respond ONLY with a valid JSON object matching the exact key structure of the current CV input. No markdown wrappers outside the JSON.

</system_instructions>

<job_description>
{job_description}
</job_description>



<current_cv_summary>
{current_cv_summary}
</current_cv_summary>

<detailed_experience_and_projects>
{detailed_experience_and_projects}
</detailed_experience_and_projects>

<output_schema>
{{
  "experience": [
    {{
      "title": "...",
      "bullets": ["...", "..."]
    }}
  ]
}}
</output_schema>
""")



# Format input values
formatted_prompt = prompt_template.format(
    job_description=job_description,

    current_cv_summary=json.dumps(current_cv_summery, indent=2),
    detailed_experience_and_projects=profile
)

# Execute query
response = llm.invoke(formatted_prompt)

print("=== THINKING ===")
print(response.additional_kwargs.get("reasoning_content"))
print("=== ANSWER ===")
print(response.content)

In [ ]:
import json
from langchain_deepseek import ChatDeepSeek
from langchain_core.prompts import PromptTemplate

# Initialize model
MODEL_NAME = "deepseek-flash"  # or "deepseek-reasoner"
TEMPERATURE = 0

llm = ChatDeepSeek(
    model=MODEL_NAME,
    temperature=TEMPERATURE,
    api_key=DEEPSEEK_API_KEY
)

# DeepSeek-Optimized Prompt Template
prompt_template = PromptTemplate.from_template("""
<system_instructions>
You are an expert Executive Technical Resume Strategist. 
Your task is to rewrite a candidate's CV components (Title, Profile Summary, Skills, and Role/Experience) to match a target Job Description.

CRITICAL RULES:
1. TRUTHFUL RE-FRAMING: Do NOT invent experience or tools not present in the source. Re-frame real experience or add experience which is present  or relevent in the profile

2. LENGTH CONSTRAINTS (CRITICAL): 
   - The Profile Summary MUST be highly concise (maximum 100-120 words). 
   - You MUST consolidate the Skills section into a MAXIMUM of 6 or 7 distinct categories. try to match the given skills size.

3. STRATEGIC BOLDING: Use Markdown bolding (`**text**`) VERY sparingly to draw the recruiter's eye to the most critical information. 

4. OUTPUT FORMAT: Respond ONLY with a valid JSON object matching the exact key structure of the current CV input. No markdown wrappers outside the JSON.

</system_instructions>

<job_description>
{job_description}
</job_description>

<target_skills>
{target_skills}
</target_skills>

<current_cv_summary>
{current_cv_summary}
</current_cv_summary>

<detailed_experience_and_projects>
{detailed_experience_and_projects}
</detailed_experience_and_projects>

<output_schema>
{{
  "title": "...",
  "profile": "...",
  "skills": [
    {{
      "label": "...",
      "items": "..."
    }}
  ],
  "role": "..."
}}
</output_schema>
""")



# Format input values
formatted_prompt = prompt_template.format(
    job_description=job_description,
    target_skills=json.dumps(job , indent=2),
    current_cv_summary=json.dumps(current_cv_summery, indent=2),
    detailed_experience_and_projects=profile
)

# Execute query
response = llm.invoke(formatted_prompt)

print("=== THINKING ===")
print(response.additional_kwargs.get("reasoning_content"))
print("=== ANSWER ===")
print(response.content)

In [ ]:
import copy
import json
import re

import docx
from docx.text.paragraph import Paragraph


# ============================================================================
# HELPERS  (you never call these directly)
# ============================================================================

def is_bullet(p):
    """Is this paragraph a bullet? We look at the XML numbering, not the style
    name, because two of your bullets use the style 'No Spacing'."""
    return p._p.pPr is not None and p._p.pPr.numPr is not None


def get_section(doc, header):
    """All non-empty paragraphs between the ALL-CAPS line `header`
    (e.g. 'PROFILE') and the next ALL-CAPS line."""
    inside = False
    result = []
    for p in doc.paragraphs:
        text = p.text.strip()
        is_header = len(text) > 2 and text.isupper() and not is_bullet(p)
        if is_header:
            if inside:                      # reached the NEXT header -> done
                break
            inside = (text == header)       # start collecting after OUR header
        elif inside and text:
            result.append(p)
    return result


def read_text(p):
    """Paragraph -> text, with bold parts wrapped in **...**."""
    out = ""
    for run in p.runs:
        if run.bold and run.text.strip():
            out += "**" + run.text + "**"
        else:
            out += run.text
    return out.replace("****", "").strip()   # "**a****b**" -> "**ab**"


def set_text(p, new_text):
    """Replace the WHOLE text of one paragraph, keeping its look.
    Style / bullet / spacing are untouched because we only replace the runs."""
    # 1. remember the font settings of one existing run (prefer a non-bold one)
    template = None
    for run in p.runs:
        if run.text.strip() and not run.bold:
            template = run
            break
    if template is None and p.runs:
        template = p.runs[0]
    font_xml = None
    if template is not None and template._r.rPr is not None:
        font_xml = copy.deepcopy(template._r.rPr)

    # 2. delete every old run
    for run in p.runs:
        run._r.getparent().remove(run._r)

    # 3. add new runs: "aaa **bbb** ccc" -> "aaa " | "**bbb**" | " ccc"
    for chunk in re.split(r"(\*\*.+?\*\*)", new_text):
        if chunk == "":
            continue
        bold = chunk.startswith("**")
        run = p.add_run(chunk[2:-2] if bold else chunk)   # drop the ** at both ends
        if font_xml is not None:
            run._r.insert(0, copy.deepcopy(font_xml))    # same font as before
        if bold:
            run.bold = True


def make_count(paragraphs, n):
    """Return exactly n paragraphs. Need more? copy the last one.
    Have too many? delete from the end."""
    paragraphs = list(paragraphs)
    while len(paragraphs) < n:
        last = paragraphs[-1]
        new_el = copy.deepcopy(last._p)
        last._p.addnext(new_el)
        paragraphs.append(Paragraph(new_el, last._parent))
    while len(paragraphs) > n:
        gone = paragraphs.pop()
        gone._p.getparent().remove(gone._p)
    return paragraphs


def split_experience(doc):
    """The PROFESSIONAL EXPERIENCE section as
         job_header  (one paragraph)
         projects    [ {"title": <paragraph>, "bullets": [<paragraph>, ...]}, ... ]
    Rule: a bullet belongs to the last title seen; a non-bullet line is a new title."""
    paras = get_section(doc, "PROFESSIONAL EXPERIENCE")
    job_header = paras[0]
    projects = []
    for p in paras[1:]:
        if is_bullet(p):
            projects[-1]["bullets"].append(p)
        else:
            projects.append({"title": p, "bullets": []})
    return job_header, projects


# ============================================================================
# 1. GRAB EVERYTHING
# ============================================================================

def grab_all(doc):
    cv = {}

    cv["title"] = read_text(doc.paragraphs[1])                 # line under your name
    cv["profile"] = read_text(get_section(doc, "PROFILE")[0])

    cv["skills"] = []
    for p in get_section(doc, "KEY SKILLS"):
        label, items = p.text.split(":", 1)                    # "Label:  a, b, c"
        cv["skills"].append({"label": label.strip(), "items": items.strip()})

    job_header, projects = split_experience(doc)
    cv["role"] = job_header.text.split("—")[0].strip()         # "AI Engineer"
    cv["experience"] = []
    for proj in projects:
        cv["experience"].append({
            "title": proj["title"].text.strip(),
            "bullets": [read_text(b) for b in proj["bullets"]],
        })
    return cv